# R07 · Test suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<your-account>/nids-partition-distortion-ae-dae/blob/main/notebooks/reproduction/R07_Tests.ipynb)

Runs the repository's test suite (the files in `tests/`) inside the notebook. Run R00, or R01–R05, first, because some tests check the files those notebooks write. Expected without archives: **82 passed, 15 skipped** (4 need the prediction archives; 11 check contract files you have not yet copied into `configs/`).

**How to run:** Runtime → Run all. The first run in a fresh Colab runtime installs the pinned library versions and restarts the runtime once; then choose Run all again.

<details><summary>Source and conversion notes</summary>

Converted from tests/*.py; tests/conftest.py is replaced by the runner cell.

Mechanical changes made when copying the .py files into this notebook:
`from src... import ...` lines are removed (the library code is defined in the cells above);
`sys.path.insert(...)` and `from __future__ import annotations` are removed;
`Path(__file__)...` repository roots are replaced by `REPO_ROOT` from the setup cell;
the command-line interface of `src/utils/hashing.py` is omitted;
command-line arguments of `reproduce_final_statistics.py` are replaced by the parameter cell;
`sys.exit(L.finish())` becomes `assert L.finish() == 0`.
</details>

In [ ]:
# ===== Setup: run this cell first =====
# REPO_SOURCE = "drive": the repository is in your Google Drive, either as the unzipped folder
#   `nids-partition-distortion-ae-dae` or as the zip file `nids-partition-distortion-ae-dae.zip`, anywhere
#   in My Drive (up to four folders deep). The cell finds it; set DRIVE_REPO only to point at one copy.
# REPO_SOURCE = "github": after the repository is on GitHub (fill in GITHUB_URL).
REPO_SOURCE = "drive"                                                    # "drive" or "github"
DRIVE_REPO = "/content/drive/MyDrive/nids-partition-distortion-ae-dae"   # optional hint
GITHUB_URL = "https://github.com/<your-account>/nids-partition-distortion-ae-dae.git"

import os, subprocess, sys, zipfile
from importlib.metadata import version
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "nids-partition-distortion-ae-dae"


def _is_repo(p: Path) -> bool:
    return (p / "artifacts" / "ARTIFACT_PROVENANCE.csv").is_file()


def _find_repo(start: Path) -> Path:
    for p in [start, *start.parents]:
        if _is_repo(p):
            return p
    raise FileNotFoundError(f"repository folder not found above {start}")


def _locate_in_drive(hint: Path, drive_root: Path) -> Path:
    # 1. the hint itself, a folder above it, or a nested copy below it (unzipping often creates name/name/)
    for c in [hint, *hint.parents, hint / REPO_NAME]:
        if _is_repo(c):
            return c
    # 2. the unzipped folder anywhere in My Drive, up to four levels deep
    for depth in range(0, 4):
        pattern = "/".join(["*"] * depth + ["artifacts", "ARTIFACT_PROVENANCE.csv"])
        hits = sorted(drive_root.glob(pattern))
        if hits:
            if len(hits) > 1:
                print("Several copies found; using the first (set DRIVE_REPO to choose):", *[h.parents[1] for h in hits], sep="\n  ")
            return hits[0].parents[1]
    # 3. the zip file anywhere in My Drive (Google Drive does not unzip uploads): extract it to /content
    for depth in range(0, 4):
        zips = sorted(drive_root.glob("/".join(["*"] * depth + [REPO_NAME + "*.zip"])))
        if zips:
            dest = Path("/content/repo_from_zip")
            print(f"Found {zips[0]} - extracting to {dest} (results will be written there, not to Drive)")
            with zipfile.ZipFile(zips[0]) as z:
                z.extractall(dest)
            return _locate_in_drive(dest, dest)
    near = [p.name for p in drive_root.iterdir() if "nids" in p.name.lower() or "partition" in p.name.lower()] if drive_root.exists() else []
    raise FileNotFoundError(
        "Could not find the repository in Google Drive.\n"
        f"Looked for '{REPO_NAME}/artifacts/ARTIFACT_PROVENANCE.csv' (up to 4 folders deep) and for '{REPO_NAME}*.zip' in {drive_root}.\n"
        f"Items in My Drive whose names look related: {near or 'none'}.\n"
        "Fix: upload the zip file (or the unzipped folder) to My Drive, or set DRIVE_REPO to the folder that "
        "contains README.md, artifacts/ and notebooks/.")


_previous = globals().get("REPO_ROOT")                 # set by an earlier notebook in the same runtime (R00)
if _previous is not None and _is_repo(Path(_previous)):
    REPO_ROOT = Path(_previous)
elif IN_COLAB and REPO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = _locate_in_drive(Path(DRIVE_REPO), Path("/content/drive/MyDrive"))
elif IN_COLAB:
    target = Path("/content/" + REPO_NAME)
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(target)], check=True)
    REPO_ROOT = _find_repo(target)
else:                                     # local Jupyter: the notebook lives in notebooks/reproduction/
    REPO_ROOT = _find_repo(Path.cwd().resolve())
os.chdir(REPO_ROOT)

# The results were verified with these exact versions (the versions recorded by the Colab analysis).
PINNED = {"numpy": "2.1.3", "pandas": "2.2.3", "scipy": "1.16.3", "scikit-learn": "1.6.1",
          "matplotlib": "3.10.0", "pytest": "8.3.5"}
wrong = {k: v for k, v in PINNED.items() if version(k) != v}
if wrong and IN_COLAB:
    print("Installing the pinned versions:", wrong)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements.txt")], check=True)
    print("Done. The runtime will now restart; then run the notebook again from the top (Runtime > Run all).")
    os.kill(os.getpid(), 9)
elif wrong:
    print("WARNING: library versions differ from requirements.txt:", wrong)
print("Repository:", REPO_ROOT)
print("Python", sys.version.split()[0], "|", ", ".join(f"{k} {version(k)}" for k in PINNED))

## Parameters

In [ ]:
ARCHIVES = None   # optional: Drive folder with the 36 prediction archives (enables 4 more tests)

## Library code
Verbatim copies of the repository modules this notebook uses (`src/`). Run these cells; there is nothing to edit.

In [ ]:
# ===== src/utils/hashing.py =====
"""Hashing utilities that reproduce the identity conventions of the frozen Experiment 1 v2 artefacts.

Two conventions were recovered from the artefacts themselves and are checked by
``tests/test_manifest_hashes.py``:

* ``config_hash`` = SHA-256 of the canonical JSON of ``effective_config``.
* Manifest self-hashes (``partition_policy_hash``, ``selected_models_manifest_hash``,
  ``manifest_hash``, ``diagnostic_manifest_hash``, ``implementation_note_hash``, ``policy_hash``)
  = SHA-256 of the canonical JSON of the manifest without the hash field itself and without
  ``created_at_utc``.

Canonical JSON means ``json.dumps(obj, sort_keys=True, separators=(",", ":"))`` encoded as UTF-8.

Command line::

    python -m src.utils.hashing sha256 FILE [FILE ...]
    python -m src.utils.hashing manifest DIRECTORY > SHA256SUMS.txt
"""

import argparse
import hashlib
import json
from pathlib import Path
from typing import Iterable

EXCLUDED_FROM_SELF_HASH = ("created_at_utc",)


def sha256_file(path: str | Path, chunk: int = 1 << 20) -> str:
    """Return the hexadecimal SHA-256 digest of a file's bytes."""
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def canonical_json(obj) -> str:
    """Canonical JSON text used by the Experiment 1 v2 notebooks."""
    return json.dumps(obj, sort_keys=True, separators=(",", ":"))


def canonical_json_hash(obj) -> str:
    """SHA-256 of the canonical JSON text of ``obj``."""
    return hashlib.sha256(canonical_json(obj).encode("utf-8")).hexdigest()


def manifest_self_hash(manifest: dict, hash_field: str,
                       excluded: Iterable[str] = EXCLUDED_FROM_SELF_HASH) -> str:
    """Recompute a manifest's self-hash: canonical JSON without ``hash_field`` and ``excluded`` keys."""
    drop = set(excluded) | {hash_field}
    return canonical_json_hash({k: v for k, v in manifest.items() if k not in drop})


def config_hash(effective_config: dict) -> str:
    """Recompute ``config_hash`` from an ``effective_config`` block."""
    return canonical_json_hash(effective_config)

In [ ]:
# ===== src/utils/io_utils.py =====
"""Paths and loaders shared by the reproduction scripts and tests."""

import json
import os
from pathlib import Path

import pandas as pd

REPO_ROOT = Path(REPO_ROOT)  # set by the setup cell
ARTIFACTS = REPO_ROOT / "artifacts"
RESULTS = REPO_ROOT / "results"
DATASETS = {"cicids2017": "CICIDS2017", "unsw_nb15": "UNSW-NB15"}

# Columns of every observation-level prediction archive written by NB-03/NB-07.
ARCHIVE_COLUMNS = ["row_id", "y_true", "reconstruction_error", "threshold", "y_pred", "model", "seed",
                   "config_hash", "run_hash", "attack_family", "feature_fingerprint"]
ARCHIVE_DTYPES = {"row_id": str, "y_true": "int8", "reconstruction_error": "float64", "threshold": "float64",
                  "y_pred": "int8", "model": str, "seed": "int64", "config_hash": str, "run_hash": str,
                  "attack_family": str, "feature_fingerprint": str}
# CICIDS2017 web-attack family names contain U+FFFD where the original CSVs had a cp1252 en dash
# ("Web Attack � Brute Force"); the character is kept as stored in the archives.

# Environment variable that points to the folder holding the external prediction archives.
ARCHIVE_ENV = "NIDS_PREDICTION_ARCHIVES"


def art(*parts: str) -> Path:
    """Path inside ``artifacts/``."""
    return ARTIFACTS.joinpath(*parts)


def load_json(path: str | Path) -> dict:
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def read_csv(path: str | Path, **kw) -> pd.DataFrame:
    """``pandas.read_csv`` with exact (round-trip) float parsing, so that committed tables compare exactly."""
    kw.setdefault("float_precision", "round_trip")
    return pd.read_csv(path, **kw)


def read_prediction_archive(path: str | Path) -> pd.DataFrame:
    """Read one observation-level prediction archive with the dtypes used in the study."""
    return pd.read_csv(path, usecols=ARCHIVE_COLUMNS, dtype=ARCHIVE_DTYPES)


def archive_dir(cli_value: str | None = None) -> Path | None:
    """Folder with the external prediction archives (CLI argument first, then the environment variable)."""
    value = cli_value or os.environ.get(ARCHIVE_ENV)
    if not value:
        return None
    p = Path(value).expanduser()
    return p if p.is_dir() else None


def dataset_of_row_id(row_id: str) -> str:
    """``row_id`` is ``<dataset>::<source file>::<10-digit source row>``; return the repository dataset key."""
    prefix = row_id.split("::", 1)[0]
    return {"CICIDS2017": "cicids2017", "UNSWNB15": "unsw_nb15"}[prefix]


def role_sizes(ds: str) -> dict:
    """Evaluation-role sizes recorded in the frozen result tables (TP+FP+FN+TN of any run)."""
    fin = read_csv(art(ds, "final_evaluation", "final_seed_results.csv")).iloc[0]
    dev = read_csv(art(ds, "model_selection", "confirmation_seed_results.csv")).iloc[0]
    tot = lambda r: int(r.TP + r.FP + r.FN + r.TN)
    return {"final_test": tot(fin), "development": tot(dev)}


def write_csv(df: pd.DataFrame, path: str | Path, **kw) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, lineterminator="\n", **kw)
    return path

In [ ]:
# ===== src/utils/seeds.py =====
"""Seed register of Experiment 1 v2.

Every constant below is read back from a frozen artefact by ``tests/test_seed_register.py``;
the module does not set any random state for model training (training ran in the Colab notebooks).
"""
FINAL_SEEDS = (101, 202, 303, 404, 505)          # final_seed_results.csv
CONFIRMATION_SEEDS = (101, 202, 303)             # selected_models_manifest.json["confirmation_seeds"]
PARTITION_SEED = 20260914                        # partition_policy_manifest.json / integrity reports
BOOTSTRAP_SEED = 2026                            # paired_seed_bootstrap_effects.csv["bootstrap_seed"]
BOOTSTRAP_ITERATIONS = 10_000                    # paired_seed_bootstrap_effects.csv["bootstrap_iterations"]
CONFIDENCE_LEVEL = 0.95                          # paired_seed_bootstrap_effects.csv["confidence_level"]

In [ ]:
# ===== src/utils/claims.py =====
"""Record manuscript claims next to the values recomputed from the artefacts.

Each check compares a recomputed number with the value *as printed in the manuscript*. The tolerance is
half a unit in the last printed decimal place, so "0.7094" passes for any value in [0.70935, 0.70945].
Every reproduction script writes its log to ``results/claims_check/<script>.csv`` and exits with a
non-zero status if any check fails.
"""

import math
import sys
from pathlib import Path

import pandas as pd



def _parse(printed: str) -> tuple[float, float]:
    s = str(printed).replace(",", "").replace("%", "").replace("+", "").replace("−", "-").strip()
    dec = len(s.split(".")[1]) if "." in s and "e" not in s.lower() else 0
    if "e" in s.lower():
        mant = s.lower().split("e")[0]
        exp = int(s.lower().split("e")[1])
        dec_m = len(mant.split(".")[1]) if "." in mant else 0
        return float(s), 0.5 * 10 ** (exp - dec_m)
    return float(s), 0.5 * 10 ** (-dec)


class ClaimLog:
    def __init__(self, script: str):
        self.script = script
        self.rows: list[dict] = []

    def check(self, claim_id: str, location: str, description: str, reproduced, printed: str,
              source: str, scale: float = 1.0) -> bool:
        """``scale`` converts the reproduced value to the printed unit (e.g. 100 for percentages)."""
        value, tol = _parse(printed)
        rep = float(reproduced) * scale
        ok = math.isclose(rep, value, abs_tol=tol + 1e-12)
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": printed, "reproduced_value": repr(round(rep, 10)),
                          "status": "PASS" if ok else "FAIL", "source": source})
        return ok

    def assert_true(self, claim_id: str, location: str, description: str, condition: bool, source: str) -> bool:
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": "true", "reproduced_value": str(bool(condition)),
                          "status": "PASS" if condition else "FAIL", "source": source})
        return bool(condition)

    def note(self, claim_id: str, location: str, description: str, value: str, source: str, status: str) -> None:
        """Record a finding that is not a pass/fail comparison (e.g. a discrepancy that the manuscript notes)."""
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": "", "reproduced_value": value, "status": status, "source": source})

    def finish(self) -> int:
        df = pd.DataFrame(self.rows)
        out = RESULTS / "claims_check" / f"{self.script}.csv"
        out.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out, index=False, lineterminator="\n")
        n_fail = int((df.status == "FAIL").sum())
        n_pass = int((df.status == "PASS").sum())
        print(f"[{self.script}] {n_pass} checks passed, {n_fail} failed, {len(df) - n_pass - n_fail} notes -> {out}")
        if n_fail:
            print(df[df.status == "FAIL"].to_string(index=False), file=sys.stderr)
        return 1 if n_fail else 0

In [ ]:
# ===== src/fingerprinting/multiplicity_audit.py =====
"""Fingerprint-multiplicity bands, total-variation distance and related structural quantities.

The eight frozen bands are those of the UNSW-NB15 source-multiplicity amendment
(``source_multiplicity_amendment.json["bands"]``): 1, 2, 3-4, 5-9, 10-19, 20-49, 50-99, >=100.
"""

import numpy as np
import pandas as pd

BANDS = ["1", "2", "3-4", "5-9", "10-19", "20-49", "50-99", ">=100"]
EDGES = [0, 1, 2, 4, 9, 19, 49, 99, np.inf]   # right-closed intervals (0,1], (1,2], (2,4], ...


def band_of(group_size) -> pd.Series:
    """Map fingerprint-group sizes (multiplicities) to the frozen band labels."""
    return pd.cut(pd.Series(group_size), bins=EDGES, labels=BANDS, right=True).astype(str)


def shares(rows_by_band: pd.Series) -> pd.Series:
    """Band shares of rows, in the frozen band order."""
    r = rows_by_band.reindex(BANDS).fillna(0).astype(float)
    return r / r.sum()


def tv_distance(p: pd.Series, q: pd.Series) -> float:
    """Total-variation distance 1/2 * sum_b |p_b - q_b| between two band-share vectors."""
    return float(0.5 * (shares(p) - shares(q)).abs().sum())


def gate_rows(gate: pd.DataFrame, column: str = "actual_rows") -> pd.DataFrame:
    """Role x band row matrix from ``multiplicity_gate.csv``."""
    return gate.pivot(index="partition", columns="size_band", values=column)[BANDS]


def expected_cross_side_sharing(group_sizes: np.ndarray) -> float:
    """Expected number of rows sharing their fingerprint with >=1 row on the opposite side when every row
    is assigned independently to one of two sides with probability 1/2: sum_g m_g (1 - 2^{-(m_g - 1)})."""
    m = np.asarray(group_sizes, dtype=float)
    return float((m * (1.0 - np.power(0.5, m - 1.0))).sum())


def redundancy_rate(rows: int, groups: int) -> float:
    """(rows - groups) / rows: the share of rows that repeat a model-view vector already present."""
    return (rows - groups) / rows

In [ ]:
# ===== src/fingerprinting/duplicate_audit.py =====
"""Model-view redundancy accounting from the frozen PRE-00 fingerprint audits and group manifests.

The fingerprints themselves cannot be recomputed here (that requires the raw master CSVs and the
PRE-00 notebook); this module recomputes every *count* the manuscript derives from the frozen audit
outputs, and documents a property of the stored fingerprint strings.
"""

import hashlib

import numpy as np
import pandas as pd


def audit_counts(audit: dict) -> dict:
    """Corpus-level redundancy quantities from ``fingerprint_audit.json`` (Table 6 rows)."""
    n = audit["total_observations"]
    g = audit["number_of_fingerprint_groups"]
    nonsingleton_rows = (audit["number_of_observations_in_same_label_duplicate_groups"]
                         + audit["number_of_observations_in_mixed_label_groups"])
    return {
        "master_rows": n,
        "fingerprint_groups": g,
        "singleton_groups": audit["number_of_unique_fingerprint_groups"],
        "same_label_duplicate_groups": audit["number_of_same_label_duplicate_groups"],
        "same_label_duplicate_rows": audit["number_of_observations_in_same_label_duplicate_groups"],
        "mixed_label_groups": audit["number_of_mixed_label_fingerprint_groups"],
        "mixed_label_rows": audit["number_of_observations_in_mixed_label_groups"],
        "rows_in_nonsingleton_groups": nonsingleton_rows,
        "rows_in_nonsingleton_groups_pct": 100.0 * nonsingleton_rows / n,
        "modelview_duplicates_beyond_first": n - g,
        "modelview_duplicates_beyond_first_pct": 100.0 * (n - g) / n,
        "largest_group": audit["maximum_fingerprint_group_size"],
        "collision_groups_resolved": audit["resolved_effective_128bit_collision_groups"],
        "groups_reconcile": audit["number_of_unique_fingerprint_groups"] + audit["number_of_same_label_duplicate_groups"]
        + audit["number_of_mixed_label_fingerprint_groups"] == g,
        "rows_reconcile": audit["number_of_unique_fingerprint_groups"] + nonsingleton_rows == n,
    }


def error_floor(benign_rows: np.ndarray, attack_rows: np.ndarray) -> int:
    """Irreducible error of any deterministic detector on mixed-label groups: sum_g min(benign_g, attack_g)."""
    return int(np.minimum(np.asarray(benign_rows), np.asarray(attack_rows)).sum())


def pure_benign_reconciliation(gate: pd.DataFrame, duplicate_manifest: pd.DataFrame) -> dict:
    """Pure-benign rows and fingerprints of UNSW-NB15 from the v3.4 multiplicity gate and the same-label
    duplicate manifest. Singletons are counted once per row; each benign duplicate group once."""
    rows = int(gate.actual_rows.sum())
    singletons = int(gate.loc[gate.size_band == "1", "actual_rows"].sum())
    benign_dup = duplicate_manifest[duplicate_manifest.binary_label_max == 0]
    dup_rows = int(benign_dup.fingerprint_group_size.sum())
    return {"pure_benign_rows": rows, "pure_benign_singletons": singletons,
            "pure_benign_duplicate_groups": len(benign_dup), "pure_benign_duplicate_rows": dup_rows,
            "pure_benign_fingerprints": singletons + len(benign_dup),
            "reconciles": rows - singletons == dup_rows}


def fingerprint_halves_identical(fingerprints: pd.Series) -> float:
    """Share of ``h128:`` fingerprints whose two 64-bit halves are identical.

    pandas ``hash_pandas_object`` applies ``hash_key`` only to object (string) columns, so for an
    all-numeric model view (CICIDS2017) two "independently keyed" 64-bit hashes coincide and the
    candidate key carries 64 effective bits. Group membership was verified by byte equality in
    PRE-00, so the group counts are unaffected; the "128 effective bits" description is not.
    """
    h = fingerprints.astype(str).str.removeprefix("h128:")
    return float((h.str[:16] == h.str[16:32]).mean())


def tie_break_key(partition_seed: int, fingerprint: str) -> str:
    """Deterministic tie-breaking key of the CICIDS2017 partition policy: SHA-256(str(seed) + '|' + fingerprint)."""
    return hashlib.sha256(f"{partition_seed}|{fingerprint}".encode("utf-8")).hexdigest()

In [ ]:
# ===== src/partitioning/partition_integrity.py =====
"""Integrity and accounting checks on the frozen partition artefacts.

The allocation itself (fingerprint-atomic heuristic for CICIDS2017; source- and multiplicity-aware
MILP for UNSW-NB15) needs the full membership and raw data and is not re-run here. These functions
re-check the properties the manuscript reports, using the frozen gate tables, the integrity reports
and the first 100,000 rows of each membership file.
"""

import pandas as pd


BENIGN_ONLY_ROLES = ("benign_weight_train", "benign_early_stop", "benign_threshold_calibration")
EVALUATION_ROLES = ("development", "final_test")
ROLES = BENIGN_ONLY_ROLES + EVALUATION_ROLES


def membership_checks(m: pd.DataFrame) -> dict:
    """Checks on a membership table (full file or the first-100k sample)."""
    states = m.groupby("feature_fingerprint")["partition"].nunique()
    benign_only = m[m.partition.isin(BENIGN_ONLY_ROLES)]
    mixed = m[m.fingerprint_group_class.str.startswith("mixed")]
    return {
        "rows": len(m),
        "row_id_unique": bool(m.row_id.is_unique),
        "fingerprints_in_more_than_one_state": int((states > 1).sum()),
        "attack_rows_in_benign_only_roles": int((benign_only.binary_label != 0).sum()),
        "mixed_group_rows_in_benign_only_roles": int(benign_only.fingerprint_group_class.str.startswith("mixed").sum()),
        "mixed_group_states": sorted(mixed.partition.unique().tolist()),
    }


def role_capacity_checks(gate: pd.DataFrame, integrity: dict) -> dict:
    """UNSW-NB15 v3.4: gate role totals versus the capacities in the integrity report."""
    totals = gate.groupby("partition").actual_rows.sum()
    cap = integrity["role_pure_benign_capacities"]
    return {"role_totals_equal_capacities": all(int(totals[r]) == int(cap[r]) for r in cap),
            "total_equals_pure_benign_rows": int(totals.sum()) == integrity["pure_benign_rows"],
            "all_cells_pass": bool(gate["pass"].all()),
            "max_abs_deviation": int(gate.absolute_deviation.max()),
            "cells_at_max_deviation": gate.loc[gate.absolute_deviation == gate.absolute_deviation.max(),
                                               ["partition", "size_band", "absolute_deviation", "tolerance"]]
            .to_dict("records")}


def source_checks(source_actual: pd.DataFrame, singleton_gate: pd.DataFrame, sxm: pd.DataFrame,
                  gate: pd.DataFrame) -> dict:
    """Cross-table consistency of the UNSW-NB15 source gates with the multiplicity gate."""
    by_role = source_actual.set_index("partition").sum(axis=1)
    g_role = gate.groupby("partition").actual_rows.sum()
    sing = singleton_gate.groupby("partition").actual_rows.sum()
    g_sing = gate[gate.size_band == "1"].set_index("partition").actual_rows
    sxm_rb = sxm.groupby(["partition", "size_band"]).actual_rows.sum()
    g_rb = gate.set_index(["partition", "size_band"]).actual_rows
    nonexempt = sxm[~sxm.is_exempt]
    return {"source_totals_equal_gate_totals": bool((by_role.reindex(g_role.index) == g_role).all()),
            "singleton_source_equals_gate_band1": bool((sing.reindex(g_sing.index) == g_sing).all()),
            "source_x_band_sums_equal_gate": bool((sxm_rb.reindex(g_rb.index) == g_rb).all()),
            "nonexempt_cells": len(nonexempt), "nonexempt_cells_pass": bool(nonexempt.pass_nonexempt.all()),
            "exempt_cells": sxm.loc[sxm.is_exempt, ["source_file", "size_band"]].drop_duplicates().to_dict("records")}


def band_ge100_checks(band_report: pd.DataFrame, gate: pd.DataFrame) -> dict:
    g = gate[gate.size_band == ">=100"].set_index("partition").actual_rows
    b = band_report.set_index("partition")
    return {"rows_equal_gate": bool((b.rows.reindex(g.index) == g).all()),
            "groups_total": int(b.groups.sum()), "rows_total": int(b.rows.sum()),
            "largest_group_by_role": b.max_group_size.astype(int).to_dict()}


def cicids_integrity_summary(report: dict) -> dict:
    checks = report["hard_validation_results"]
    return {"status": report["final_status"], "hard_checks": len(checks),
            "hard_checks_passed": sum(bool(c["passed"]) for c in checks if c["hard"]),
            "blocking_failures": len(report["blocking_failures"]),
            "development_attack_prevalence": report["observed_allocation"]["development_attack_prevalence"],
            "final_test_attack_prevalence": report["observed_allocation"]["final_test_attack_prevalence"],
            "attack_reserve_rows": report["observed_allocation"]["attack_reserve_rows"]}

In [ ]:
# ===== src/preprocessing/preprocessing_integrity.py =====
"""Checks of the frozen preprocessing hand-off: fit scope and model-input column order.

Preprocessing itself (PRE-02: imputation, scaling, encoding) ran in the Colab notebooks and its fitted
objects are not part of the released evidence; only its contract can be checked here.
"""

import pandas as pd

# Raw UNSW-NB15 columns that the frozen model view excludes (identifiers, timestamps and labels),
# as listed in Table 3 of the manuscript.
UNSW_EXCLUDED_RAW = ["srcip", "sport", "dstip", "dsport", "Stime", "Ltime", "attack_cat", "Label"]


def fit_scope(handoff: dict) -> dict:
    fr = handoff["fit_rule"]
    return {"fit_partition": fr["learned_preprocessing_fit_partition"],
            "fit_on_any_other_partition": fr["fit_on_any_other_partition"],
            "transform_partitions": handoff["partition_names"],
            "final_test_fit_prohibited": any("fit(final_test)" in s for s in handoff["final_test_prohibited_before_notebook_07"])}


def cicids_column_order_matches(handoff: dict, audit: dict) -> bool:
    """The model-input order handed to PRE-02 equals the order that was fingerprinted in PRE-00."""
    return handoff["model_input_column_order"] == audit["algorithm"]["column_order"]


def unsw_model_view_from_schema(schema: pd.DataFrame) -> list[str]:
    """The 49-column raw schema minus the excluded columns, in file order, with names as stored."""
    keep = ~schema.assigned_column_name.isin(UNSW_EXCLUDED_RAW)
    return schema.loc[keep, "name_as_stored_in_feature_file"].tolist()

In [ ]:
# ===== src/models/architecture.py =====
"""Layer specification of the three candidate autoencoder architectures (manuscript Section 3.9).

    A: p -> 32 -> 16 -> z -> 16 -> 32 -> p
    B: p -> 64 -> 32 -> z -> 32 -> 64 -> p
    C: p -> 64 -> 32 -> 16 -> z -> 16 -> 32 -> 64 -> p

with z = latent_fraction x (final encoder width), fully connected layers with biases, ReLU hidden
activations and a linear output. This module is a *specification*, not the training code: it contains no
weights and does not train anything (training ran in the NB-01/NB-02/NB-07 Colab notebooks with PyTorch).
It reproduces the recorded latent dimension and parameter count of all 12 confirmation candidates
(``tests/test_architecture.py``), which pins down the architecture and the transformed input dimension p
(78 for CICIDS2017, 79 for UNSW-NB15).
"""

ENCODER_WIDTHS = {"A": (32, 16), "B": (64, 32), "C": (64, 32, 16)}
TRANSFORMED_INPUT_DIM = {"cicids2017": 78, "unsw_nb15": 79}


def layer_widths(architecture: str, p: int, latent_fraction: float) -> list[int]:
    enc = list(ENCODER_WIDTHS[architecture])
    z = int(round(enc[-1] * latent_fraction))
    return [p] + enc + [z] + enc[::-1] + [p]


def latent_dim(architecture: str, latent_fraction: float) -> int:
    return int(round(ENCODER_WIDTHS[architecture][-1] * latent_fraction))


def parameter_count(widths: list[int]) -> int:
    """Weights plus biases of a chain of fully connected layers."""
    return sum(a * b + b for a, b in zip(widths[:-1], widths[1:]))

In [ ]:
# ===== src/evaluation/metrics.py =====
"""Observation-level detection metrics, named as in the frozen ``final_seed_results.csv`` tables.

Positive class = attack (``y_true == 1``). Scores are reconstruction errors; a larger score is more
anomalous. ``PR-AUC`` is the trapezoidal area under the precision-recall curve
(``sklearn.metrics.auc(recall, precision)``), which is distinct from ``Average_Precision``.
Each function was checked against every frozen run in the study (see ``tests/test_archive_recomputation.py``).
"""

import numpy as np
from sklearn.metrics import (auc, average_precision_score, precision_recall_curve, roc_auc_score,
                             roc_curve)


def confusion(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y = np.asarray(y_true).astype(bool)
    p = np.asarray(y_pred).astype(bool)
    return {"TP": int((y & p).sum()), "FP": int((~y & p).sum()),
            "FN": int((y & ~p).sum()), "TN": int((~y & ~p).sum())}


def mcc_from_counts(TP: int, FP: int, FN: int, TN: int) -> float:
    den = np.sqrt(float(TP + FP) * float(TP + FN) * float(TN + FP) * float(TN + FN))
    return float((TP * TN - FP * FN) / den) if den > 0 else 0.0


def threshold_metrics(TP: int, FP: int, FN: int, TN: int) -> dict:
    """Threshold-dependent metrics from a confusion matrix."""
    rec = TP / (TP + FN)
    prec = TP / (TP + FP)
    fpr = FP / (FP + TN)
    return {"MCC": mcc_from_counts(TP, FP, FN, TN), "Recall": rec,
            "F1": 2 * prec * rec / (prec + rec), "Precision": prec, "FPR": fpr,
            "FP_per_1000_benign": 1000.0 * fpr, "Balanced_Accuracy": (rec + TN / (TN + FP)) / 2}


def ranking_metrics(y_true: np.ndarray, scores: np.ndarray) -> dict:
    """Threshold-free metrics of the reconstruction-error ranking."""
    pr, rc, _ = precision_recall_curve(y_true, scores)
    fpr, tpr, _ = roc_curve(y_true, scores)
    return {"PR-AUC": float(auc(rc, pr)), "ROC-AUC": float(roc_auc_score(y_true, scores)),
            "Average_Precision": float(average_precision_score(y_true, scores)),
            "TPR_at_1pct_FPR": float(tpr[fpr <= 0.01].max())}


def all_metrics(y_true: np.ndarray, y_pred: np.ndarray, scores: np.ndarray) -> dict:
    c = confusion(y_true, y_pred)
    out = dict(c)
    out.update(threshold_metrics(**c))
    out.update(ranking_metrics(y_true, scores))
    return out


# Metric columns shared by final_seed_results.csv and confirmation_seed_results.csv
FROZEN_METRIC_COLUMNS = ["MCC", "Recall", "F1", "Precision", "PR-AUC", "FPR", "FP_per_1000_benign",
                         "Balanced_Accuracy", "ROC-AUC", "Average_Precision", "TPR_at_1pct_FPR",
                         "TP", "TN", "FP", "FN"]

In [ ]:
# ===== src/evaluation/thresholding.py =====
"""Checks of the frozen decision rule in the prediction archives.

The decision rule recorded in every archive is ``y_pred = 1`` if and only if
``reconstruction_error > threshold`` (strict inequality), with one threshold per run. The threshold
itself is the 99th percentile of benign calibration-role reconstruction error; the calibration
errors are not part of the released evidence, so only the rule and its constancy are checked here.
The ladder check in ``tests/test_partition_accounting.py`` confirms, for the UNSW-NB15 v3.4 gate,
that the recorded threshold equals the recorded calibration 0.99 quantile.
"""

import numpy as np
import pandas as pd


def apply_threshold(scores: np.ndarray, threshold: float) -> np.ndarray:
    """Frozen rule: flag an observation as attack when its error is strictly above the threshold."""
    return (np.asarray(scores) > threshold).astype(np.int8)


def check_archive_rule(df: pd.DataFrame) -> dict:
    """Return the integrity flags of one prediction archive."""
    thr = df["threshold"].to_numpy()
    unique = np.unique(thr)
    consistent = bool((apply_threshold(df["reconstruction_error"].to_numpy(), unique[0]) ==
                       df["y_pred"].to_numpy()).all()) if unique.size == 1 else False
    return {"threshold_unique_values": int(unique.size), "threshold": float(unique[0]),
            "prediction_rule_consistent": consistent, "row_id_unique": bool(df["row_id"].is_unique)}

In [ ]:
# ===== src/evaluation/paired_analysis.py =====
"""Paired seed-level contrasts (DAE minus AE) used for Table 10 and Fig. 8 of the manuscript.

The resampling unit is the seed pair. With five pairs there are 5**5 = 3,125 equally likely ordered
bootstrap resamples, so the bootstrap distribution of the mean can be enumerated exactly; the
manuscript reports percentile limits of that exact distribution (NumPy default linear interpolation).
``bootstrap_interval`` is a Monte-Carlo percentile bootstrap with ``numpy.random.default_rng(2026)`` and
10,000 resamples, kept for comparison. It is *not* the NB-08 code (which is not in the evidence set): at six
decimals it equals the exact limits for 31 of 32 bounds (CICIDS2017 PR-AUC upper: -0.007701 vs -0.007694),
whereas the frozen NB-08 table equals them for 30 of 32 (see docs/LIMITATIONS.md).
"""

import itertools

import numpy as np
import pandas as pd
from scipy import stats



def paired_differences(values: pd.DataFrame, metric: str) -> pd.Series:
    """``values`` has columns metric, seed, AE, DAE (frozen ``paired_seed_values.csv`` layout)."""
    v = values[values.metric == metric].sort_values("seed")
    return pd.Series((v.DAE - v.AE).to_numpy(), index=v.seed.to_numpy(), name=metric)


def exact_bootstrap_means(d: np.ndarray) -> np.ndarray:
    """Means of all n**n ordered resamples with replacement of the paired differences."""
    d = np.asarray(d, dtype=float)
    idx = np.array(list(itertools.product(range(d.size), repeat=d.size)))
    return d[idx].mean(axis=1)


def exact_interval(d: np.ndarray, level: float = CONFIDENCE_LEVEL) -> tuple[float, float]:
    a = (1 - level) / 2
    lo, hi = np.percentile(exact_bootstrap_means(d), [100 * a, 100 * (1 - a)])
    return float(lo), float(hi)


def bootstrap_interval(d: np.ndarray, iterations: int = BOOTSTRAP_ITERATIONS, seed: int = BOOTSTRAP_SEED,
                       level: float = CONFIDENCE_LEVEL) -> tuple[float, float]:
    """Monte-Carlo percentile bootstrap over seed pairs with ``numpy.random.default_rng(seed)``."""
    d = np.asarray(d, dtype=float)
    rng = np.random.default_rng(seed)
    means = d[rng.integers(0, d.size, size=(iterations, d.size))].mean(axis=1)
    a = (1 - level) / 2
    lo, hi = np.percentile(means, [100 * a, 100 * (1 - a)])
    return float(lo), float(hi)


def summarise(d: np.ndarray) -> dict:
    """Descriptive and inferential summary of one vector of paired differences."""
    d = np.asarray(d, dtype=float)
    n = d.size
    mean, sd = d.mean(), d.std(ddof=1)
    sign = np.sign(mean)
    t = stats.ttest_1samp(d, 0.0)
    t_lo, t_hi = stats.t.interval(CONFIDENCE_LEVEL, n - 1, loc=mean, scale=sd / np.sqrt(n))
    lo, hi = exact_interval(d)
    return {"n_pairs": n, "mean_diff": mean, "median_diff": float(np.median(d)), "sd_diff": sd,
            "dz": mean / sd, "n_same_sign": int((np.sign(d) == sign).sum()),
            "t": float(t.statistic), "p_t": float(t.pvalue), "t_lo": float(t_lo), "t_hi": float(t_hi),
            "p_sign": float(stats.binomtest(int((d > 0).sum()), int((d != 0).sum())).pvalue),
            "p_wilcoxon": float(stats.wilcoxon(d).pvalue),
            "exact_lo": lo, "exact_hi": hi}


def contrast_table(values: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    rows = []
    for m in metrics:
        d = paired_differences(values, m).to_numpy()
        rows.append({"metric": m, **summarise(d)})
    return pd.DataFrame(rows)

In [ ]:
# ===== src/evaluation/family_diagnostics.py =====
"""Attack-family diagnostics of the final-test runs (Table S3 of the Supplementary Material).

Family figures are descriptive. They played no part in model selection
(``selected_models_manifest.json["family_metrics_used_for_selection"] is False``).
"""

import pandas as pd


def family_table(fam: pd.DataFrame) -> pd.DataFrame:
    """Pool the five final seeds per model and family.

    ``fam`` is ``family_detection_final_test.csv``. Recall per row = detected rows / rows; recall per
    vector = detected fingerprints / fingerprints (a fingerprint counts as detected when its rows are
    flagged; every fingerprint group received one prediction); miss share = the family's share of the
    model's missed attack rows.
    """
    g = fam.groupby(["model", "attack_family"]).agg(rows=("rows", "sum"), det=("detected_rows", "sum"),
                                                    fps=("fingerprints", "sum"), detfp=("detected_fingerprints", "sum"),
                                                    seeds=("seed", "nunique"), rows_per_seed=("rows", "max"),
                                                    fps_per_seed=("fingerprints", "max")).reset_index()
    g["recall_per_row"] = g.det / g.rows
    g["recall_per_vector"] = g.detfp / g.fps
    g["misses"] = g.rows - g.det
    g["miss_share_pct"] = 100 * g.misses / g.groupby("model").misses.transform("sum")
    wide = g.pivot(index="attack_family", columns="model",
                   values=["recall_per_row", "recall_per_vector", "miss_share_pct"])
    wide.columns = [f"{m}_{v}" for v, m in wide.columns]
    base = g[g.model == "AE"].set_index("attack_family")[["rows_per_seed", "fps_per_seed", "seeds"]]
    out = base.join(wide).reset_index().rename(columns={"rows_per_seed": "rows", "fps_per_seed": "distinct_vectors"})
    return out.sort_values("rows", ascending=False).reset_index(drop=True)

In [ ]:
# ===== src/evaluation/archive_summary.py =====
"""Summarise one observation-level prediction archive into the per-run quantities used in the manuscript.

Produces, for one archive:
* a run record: integrity flags, confusion counts, all frozen metrics, fingerprint-level recall and FPR,
  score geometry of missed attacks, and the largest fingerprint group present;
* per-family detection counts, per row and per distinct attack fingerprint;
* label counts of every mixed-label fingerprint group present (development archives only).

Used by ``scripts/reproduce_final_statistics.py --archives DIR``. The outputs committed under
``artifacts/<dataset>/*/derived_from_archives/`` were produced by this module from the 36 archives.
"""

from pathlib import Path

import numpy as np
import pandas as pd



def summarise_archive(path: str | Path, with_sha256: bool = True) -> dict:
    path = Path(path)
    df = read_prediction_archive(path)
    ds = dataset_of_row_id(df.row_id.iat[0])
    sizes = role_sizes(ds)
    n = len(df)
    role = {v: k for k, v in sizes.items()}.get(n, "unknown")
    y = df.y_true.to_numpy()
    p = df.y_pred.to_numpy()
    s = df.reconstruction_error.to_numpy()

    run = {"archive_file": path.name, "dataset": ds, "role": role, "model": df.model.iat[0],
           "seed": int(df.seed.iat[0]), "config_hash": df.config_hash.iat[0], "run_hash": df.run_hash.iat[0],
           "n_rows": n, "n_attack_rows": int(y.sum())}
    if with_sha256:
        run["archive_sha256"] = sha256_file(path)
        run["archive_bytes"] = path.stat().st_size
    run.update(check_archive_rule(df))
    run.update(all_metrics(y, p, s))

    # fingerprint level
    g = df.groupby("feature_fingerprint", sort=False).agg(
        lab_min=("y_true", "min"), lab_max=("y_true", "max"),
        pmin=("y_pred", "min"), pmax=("y_pred", "max"), size=("y_true", "size"))
    att = g[g.lab_min == 1]
    ben = g[g.lab_max == 0]
    run.update({"n_fingerprints": len(g), "n_mixed_label_fingerprints": int((g.lab_min != g.lab_max).sum()),
                "n_fingerprints_with_varying_prediction": int((g.pmin != g.pmax).sum()),
                "n_attack_fingerprints": len(att), "recall_per_attack_fingerprint": float((att.pmax == 1).mean()),
                "n_benign_fingerprints": len(ben), "fpr_per_benign_fingerprint": float((ben.pmax == 1).mean()),
                "largest_fingerprint_group_rows": int(g["size"].max())})
    nfam = df[df.y_true == 1].groupby("feature_fingerprint").attack_family.nunique()
    multi = nfam.index[nfam > 1]
    run["n_attack_fingerprints_with_several_families"] = int(len(multi))
    run["attack_rows_in_several_family_fingerprints"] = int(df.feature_fingerprint.isin(multi).sum())
    big = g["size"].idxmax()
    fam_big = df.loc[df.feature_fingerprint == big, "attack_family"].dropna().unique()
    run["largest_fingerprint_group_families"] = "|".join(sorted(fam_big)) if len(fam_big) else "BENIGN"

    # score geometry relative to the run threshold
    thr = run["threshold"]
    miss = s[(y == 1) & (p == 0)]
    ben_median = float(np.median(s[y == 0]))
    run.update({"benign_median_error": ben_median,
                "missed_attack_median_error_over_threshold": float(np.median(miss) / thr) if miss.size else np.nan,
                "missed_attack_p90_error_over_threshold": float(np.percentile(miss, 90) / thr) if miss.size else np.nan,
                "attack_share_below_benign_median": float((s[y == 1] < ben_median).mean()),
                "attack_share_at_or_below_threshold": float((s[y == 1] <= thr).mean())})

    # families (attack rows only)
    a = df[df.y_true == 1]
    fam_rows = a.groupby("attack_family").agg(rows=("y_pred", "size"), detected_rows=("y_pred", "sum"))
    fam_fp = a.groupby(["attack_family", "feature_fingerprint"]).y_pred.max().groupby("attack_family").agg(
        fingerprints="size", detected_fingerprints="sum")
    fam = fam_rows.join(fam_fp).reset_index()
    for k in ("dataset", "role", "model", "seed", "config_hash", "run_hash"):
        fam.insert(0, k, run[k])
    fam = fam[["dataset", "role", "model", "seed", "config_hash", "run_hash", "attack_family",
               "rows", "detected_rows", "fingerprints", "detected_fingerprints"]]

    # mixed-label groups with their label composition
    mixed_ids = g.index[g.lab_min != g.lab_max]
    m = df[df.feature_fingerprint.isin(mixed_ids)].assign(
        detected_attack=lambda x: ((x.y_true == 1) & (x.y_pred == 1)).astype(int))
    mixed = m.groupby("feature_fingerprint").agg(rows=("y_true", "size"), attack_rows=("y_true", "sum"),
                                                 attack_rows_detected=("detected_attack", "sum"))
    mixed["benign_rows"] = mixed.rows - mixed.attack_rows
    fams = m[m.y_true == 1].groupby("feature_fingerprint").attack_family.agg(lambda v: "|".join(sorted(set(v.dropna()))))
    mixed["attack_families"] = fams
    mixed = mixed.reset_index()[["feature_fingerprint", "rows", "benign_rows", "attack_rows", "attack_families",
                                 "attack_rows_detected"]]
    keep = ~df.feature_fingerprint.isin(mixed_ids).to_numpy()
    run["MCC_excluding_mixed_label_groups"] = mcc_from_counts(**confusion(y[keep], p[keep]))
    run["mixed_label_rows"] = int((~keep).sum())
    run["mixed_label_attack_rows"] = int(y[~keep].sum())
    run["mixed_label_attack_rows_detected"] = int(p[(~keep) & (y == 1)].sum())
    return {"run": run, "families": fam, "mixed_groups": mixed}


def summarise_directory(archive_dir: str | Path, out_dir: str | Path, workers: int = 1) -> dict:
    """Summarise every ``*.csv`` archive under ``archive_dir`` and write tidy tables to ``out_dir``.

    Writes ``run_summary.csv`` (one row per archive), ``family_detection.csv`` and
    ``mixed_label_groups.csv`` (one row per mixed-label group per archive). Returns the three tables.
    """
    from concurrent.futures import ProcessPoolExecutor
    files = sorted(Path(archive_dir).glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"no prediction archives (*.csv) found in {archive_dir}")
    if workers > 1:
        with ProcessPoolExecutor(workers) as ex:
            parts = list(ex.map(summarise_archive, files))
    else:
        parts = [summarise_archive(f) for f in files]
    runs = pd.DataFrame([p["run"] for p in parts]).sort_values(["dataset", "role", "model", "config_hash", "seed"])
    fams = pd.concat([p["families"] for p in parts]).sort_values(["dataset", "role", "model", "config_hash", "seed", "attack_family"])
    mixed = []
    for p in parts:
        if len(p["mixed_groups"]):
            r = p["run"]
            mixed.append(p["mixed_groups"].assign(dataset=r["dataset"], role=r["role"], model=r["model"],
                                                  seed=r["seed"], config_hash=r["config_hash"]))
    mixed = pd.concat(mixed) if mixed else pd.DataFrame()
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    runs.to_csv(out / "run_summary.csv", index=False, lineterminator="\n")
    fams.to_csv(out / "family_detection.csv", index=False, lineterminator="\n")
    mixed.to_csv(out / "mixed_label_groups.csv", index=False, lineterminator="\n")
    return {"runs": runs, "families": fams, "mixed": mixed}


def repository_tables(tables: dict) -> dict:
    """Split the directory summary into the per-dataset files committed under ``artifacts/``.

    Keys are paths relative to ``artifacts/``. Mixed-label group composition is identical in every
    development archive of a dataset (checked here), so it is stored once per dataset.
    """
    runs, fams, mixed = tables["runs"], tables["families"], tables["mixed"]
    out = {}
    for ds in sorted(runs.dataset.unique()):
        r = runs[runs.dataset == ds]
        f = fams[fams.dataset == ds]
        out[f"{ds}/final_evaluation/derived_from_archives/run_summary_final_test.csv"] = r[r.role == "final_test"]
        out[f"{ds}/final_evaluation/derived_from_archives/family_detection_final_test.csv"] = f[f.role == "final_test"]
        out[f"{ds}/model_selection/derived_from_archives/run_summary_development.csv"] = r[r.role == "development"]
        if len(mixed):
            mx = mixed[mixed.dataset == ds]
            comp = mx[["feature_fingerprint", "rows", "benign_rows", "attack_rows", "attack_families", "role"]]
            base = comp.drop_duplicates()
            if base.feature_fingerprint.duplicated().any():
                raise ValueError(f"{ds}: mixed-label group composition differs between archives")
            out[f"{ds}/fingerprints/derived_from_archives/mixed_label_group_composition.csv"] = \
                base.sort_values("feature_fingerprint").reset_index(drop=True)
    manifest = runs[["archive_file", "archive_sha256", "archive_bytes", "n_rows", "dataset", "role", "model",
                     "seed", "config_hash", "run_hash"]]
    out["../external_artifacts/prediction_archive_manifest.csv"] = manifest.sort_values(
        ["dataset", "role", "model", "config_hash", "seed"])
    return out


def write_repository_tables(tables: dict, root: str | Path) -> list[Path]:
    """Write ``repository_tables(tables)`` below ``root`` (normally the ``artifacts/`` folder)."""
    written = []
    for rel, df in repository_tables(tables).items():
        p = (Path(root) / rel).resolve()
        p.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(p, index=False, lineterminator="\n", float_format="%.17g")
        written.append(p)
    return written

## Test runner

In [ ]:
# ===== Minimal test runner (replaces pytest's collection; uses pytest only for marks and skip) =====
import inspect, traceback
import pytest
import pandas as pd

ARCHIVES_DIR = Path(ARCHIVES) if ARCHIVES else None
TEST_RESULTS = []
_RAN = set()


def _fixture(name):
    if name == "archives":
        if ARCHIVES_DIR is None or not ARCHIVES_DIR.is_dir():
            pytest.skip("external prediction archives not available (set ARCHIVES in the parameter cell)")
        return ARCHIVES_DIR
    if name == "archive_manifest":
        return pd.read_csv(REPO_ROOT / "external_artifacts" / "prediction_archive_manifest.csv")
    raise KeyError(name)


def _cases(fn):
    combos = [{}]
    for m in reversed([m for m in getattr(fn, "pytestmark", []) if m.name == "parametrize"]):
        names = [n.strip() for n in m.args[0].split(",")] if isinstance(m.args[0], str) else list(m.args[0])
        new = []
        for c in combos:
            for v in list(m.args[1]):
                v = v.values if type(v).__name__ == "ParameterSet" else v
                new.append({**c, **dict(zip(names, v if len(names) > 1 else (v,)))})
        combos = new
    return combos


def run_tests(module_name, namespace):
    tests = [(k, v) for k, v in list(namespace.items()) if k.startswith("test_") and callable(v) and id(v) not in _RAN]
    for name, fn in tests:
        _RAN.add(id(fn))
        for case in _cases(fn):
            label = ", ".join(f"{k}={str(v)[:40]}" for k, v in case.items())
            try:
                kwargs = dict(case)
                for p in inspect.signature(fn).parameters:
                    if p not in kwargs:
                        kwargs[p] = _fixture(p)
                fn(**kwargs)
                status, msg = "passed", ""
            except pytest.skip.Exception as e:
                status, msg = "skipped", str(e)
            except AssertionError as e:
                status, msg = "FAILED", (str(e) or traceback.format_exc(limit=2))[:300]
            except Exception as e:
                status, msg = "ERROR", f"{type(e).__name__}: {e}"[:300]
            TEST_RESULTS.append((module_name, name, label, status, msg))
    done = [r for r in TEST_RESULTS if r[0] == module_name]
    print(f"{module_name}: " + ", ".join(f"{s} {sum(r[3] == s for r in done)}" for s in ("passed", "skipped", "FAILED", "ERROR")))

### `tests/test_architecture.py`

In [ ]:
# ===== tests/test_architecture.py =====
import pytest



@pytest.mark.parametrize("ds", ["cicids2017", "unsw_nb15"])
def test_candidate_parameter_counts(ds):
    p = TRANSFORMED_INPUT_DIM[ds]
    ranking = read_csv(art(ds, "model_selection", "selection_ranking.csv")).set_index("config_hash")
    n = 0
    for m in ("AE", "DAE"):
        for c in load_json(art(ds, "model_selection", f"top_three_configs_{m}.json"))["configs"]:
            e = c["effective_config"]
            assert latent_dim(e["architecture"], e["latent_fraction"]) == e["latent_dim"]
            assert parameter_count(layer_widths(e["architecture"], p, e["latent_fraction"])) == ranking.loc[c["config_hash"], "parameter_count"]
            n += 1
    assert n == 6


def test_selected_models_table9():
    assert parameter_count(layer_widths("B", 78, 0.5)) == 15_390
    assert parameter_count(layer_widths("B", 79, 0.5)) == 15_519

run_tests('test_architecture', globals())

### `tests/test_archive_recomputation.py`

In [ ]:
# ===== tests/test_archive_recomputation.py =====
import numpy as np



def test_archive_checksums(archives, archive_manifest):
    for r in archive_manifest.itertuples():
        p = archives / r.archive_file
        assert p.exists(), f"missing archive {r.archive_file}"
        assert p.stat().st_size == r.archive_bytes
        assert sha256_file(p) == r.archive_sha256


def test_one_final_archive_per_corpus(archives, archive_manifest):
    for ds in ("cicids2017", "unsw_nb15"):
        row = archive_manifest[(archive_manifest.dataset == ds) & (archive_manifest.role == "final_test")].iloc[0]
        got = summarise_archive(archives / row.archive_file, with_sha256=False)["run"]
        committed = read_csv(art(ds, "final_evaluation", "derived_from_archives", "run_summary_final_test.csv"))
        c = committed[committed.archive_file == row.archive_file].iloc[0]
        frozen = read_csv(art(ds, "final_evaluation", "final_seed_results.csv"))
        f = frozen[(frozen.config_hash == got["config_hash"]) & (frozen.seed == got["seed"])].iloc[0]
        for col in FROZEN_METRIC_COLUMNS:
            assert np.isclose(got[col], c[col], rtol=0, atol=1e-12)
            assert np.isclose(got[col], f[col], rtol=0, atol=1e-12)
        assert got["recall_per_attack_fingerprint"] == c.recall_per_attack_fingerprint

run_tests('test_archive_recomputation', globals())

### `tests/test_benign_role_purity.py`

In [ ]:
# ===== tests/test_benign_role_purity.py =====
import pytest


F = ("unsw_nb15", "partition", "final_source_multiplicity_balanced")


@pytest.mark.parametrize("path", [("cicids2017", "partition", "partition_membership_first100k.csv"),
                                  F + ("partition_membership_first100k.csv",)])
def test_membership_sample_purity(path):
    r = membership_checks(read_csv(art(*path)))
    assert r["row_id_unique"]
    assert r["attack_rows_in_benign_only_roles"] == 0
    assert r["mixed_group_rows_in_benign_only_roles"] == 0
    assert set(r["mixed_group_states"]) <= {"development", "final_test"}


def test_cicids_integrity_checks_for_benign_roles():
    c = load_json(art("cicids2017", "partition", "partition_integrity_report.json"))
    chk = {x["check"]: x for x in c["hard_validation_results"]}
    for role in BENIGN_ONLY_ROLES:
        assert chk[f"{role}_contains_zero_attacks"]["passed"] and chk[f"{role}_contains_zero_attacks"]["detail"] == "attack_rows=0"
    assert chk["mixed_label_groups_only_in_development_or_final_test"]["passed"]
    assert chk["attack_reserve_contains_no_benign"]["passed"]


def test_unsw_v34_preserved_nonpure_memberships():
    u = load_json(art(*F, "partition_integrity_report.json"))
    assert u["gates"]["nonpure_membership_preserved"] is True
    # the v3.4 amendment reallocated pure-benign groups only
    assert load_json(art(*F, "source_multiplicity_amendment.json"))["scope"] == "reallocate pure benign fingerprint groups only"


def test_smoke_gates_used_benign_scores_only():
    for p in [("unsw_nb15", "partition", "source_balanced", "smoke_manifest_source_balanced.json"), F + ("smoke_manifest_final.json",)]:
        s = load_json(art(*p))
        assert s.get("attack_outcomes_accessed", False) is False and s.get("attack_scores_computed", False) is False

run_tests('test_benign_role_purity', globals())

### `tests/test_configs.py`

In [ ]:
# ===== tests/test_configs.py =====
import csv
import json

import pytest


ROWS = list(csv.DictReader(open(REPO_ROOT / "configs" / "EXPECTED_HASHES.csv", encoding="utf-8")))
COPIES = {"configs/cicids2017/partition_policy.json": "artifacts/cicids2017/partition/partition_policy_manifest.json",
          "configs/cicids2017/selected_models.json": "artifacts/cicids2017/model_selection/selected_models_manifest.json",
          "configs/unsw_nb15/multiplicity_policy.json": "artifacts/unsw_nb15/partition/final_source_multiplicity_balanced/source_multiplicity_amendment.json",
          "configs/unsw_nb15/selected_models.json": "artifacts/unsw_nb15/model_selection/selected_models_manifest.json"}


@pytest.mark.parametrize("cfg,src", COPIES.items())
def test_included_configs_are_byte_copies(cfg, src):
    assert sha256_file(REPO_ROOT / cfg) == sha256_file(REPO_ROOT / src)


def _candidate_hashes(path):
    out = {sha256_file(path)}
    try:
        obj = json.load(open(path, encoding="utf-8"))
    except (ValueError, UnicodeDecodeError):
        return out
    out.add(canonical_json_hash(obj))
    if isinstance(obj, dict):
        for k in list(obj):
            if k.endswith("hash"):
                out.add(canonical_json_hash({a: b for a, b in obj.items() if a not in (k, "created_at_utc")}))
                out.add(canonical_json_hash({a: b for a, b in obj.items() if a != k}))
    return out


@pytest.mark.parametrize("row", [r for r in ROWS if r["recorded_hash"]], ids=lambda r: r["expected_path"])
def test_config_hash_when_present(row):
    p = REPO_ROOT / row["expected_path"]
    if not p.exists():
        pytest.skip(f"{row['expected_path']} not yet copied from Google Drive (see configs/README.md)")
    assert row["recorded_hash"] in _candidate_hashes(p), f"{row['expected_path']}: no hash convention reproduces {row['recorded_hash']}"


@pytest.mark.parametrize("ds", ["cicids2017", "unsw_nb15"])
def test_data_sha256sums_record_the_master_hash(ds):
    audit = json.load(open(REPO_ROOT / "artifacts" / ds / "fingerprints" / "fingerprint_audit.json"))
    lines = [ln.split() for ln in open(REPO_ROOT / "data" / ds / "SHA256SUMS.txt") if ln.strip() and not ln.startswith("#")]
    assert lines == [[audit["master_dataset_sha256"], "MASTER_FILENAME"]]

run_tests('test_configs', globals())

### `tests/test_fingerprint_determinism.py`

In [ ]:
# ===== tests/test_fingerprint_determinism.py =====
import pandas as pd
import pytest


MEMBERSHIP = {"cicids2017": ("cicids2017", "partition", "partition_membership_first100k.csv"),
              "unsw_nb15": ("unsw_nb15", "partition", "final_source_multiplicity_balanced", "partition_membership_first100k.csv")}
MAPPING = {ds: (ds, "fingerprints", "row_modelview_group_mapping_first100k.csv") for ds in MEMBERSHIP}


@pytest.mark.parametrize("ds", list(MEMBERSHIP))
def test_mapping_and_membership_agree(ds):
    m = read_csv(art(*MEMBERSHIP[ds]), usecols=["row_id", "feature_fingerprint", "binary_label"])
    g = read_csv(art(*MAPPING[ds]), usecols=["row_id", "feature_fingerprint", "binary_label"])
    j = m.merge(g, on="row_id", suffixes=("_m", "_g"))
    assert len(j) == len(m) == len(g) == 100_000
    assert (j.feature_fingerprint_m == j.feature_fingerprint_g).all()
    assert (j.binary_label_m == j.binary_label_g).all()


@pytest.mark.parametrize("ds", list(MEMBERSHIP))
def test_fingerprint_format(ds):
    fp = read_csv(art(*MAPPING[ds]), usecols=["feature_fingerprint"]).feature_fingerprint
    assert fp.str.fullmatch(r"h128:[0-9a-f]{32}").all()


def test_unsw_group_sizes_match_duplicate_manifest():
    g = read_csv(art(*MAPPING["unsw_nb15"]))
    dup = read_csv(art("unsw_nb15", "fingerprints", "duplicate_group_manifest.csv"))
    j = g[g.fingerprint_group_class == "same-label duplicate group"].merge(dup, on="feature_fingerprint", suffixes=("", "_dup"))
    assert len(j) == (g.fingerprint_group_class == "same-label duplicate group").sum()
    assert (j.fingerprint_group_size == j.fingerprint_group_size_dup).all()
    assert (g.loc[g.fingerprint_group_class == "unique", "fingerprint_group_size"] == 1).all()


def test_cicids_fingerprints_have_64_effective_bits():
    """Documents a property of the stored strings: for the all-numeric CICIDS2017 view both 64-bit halves
    coincide (pandas ignores hash_key for numeric columns), whereas UNSW-NB15 halves differ."""
    c = read_csv(art(*MEMBERSHIP["cicids2017"]), usecols=["feature_fingerprint"]).feature_fingerprint
    u = read_csv(art(*MEMBERSHIP["unsw_nb15"]), usecols=["feature_fingerprint"]).feature_fingerprint
    assert fingerprint_halves_identical(c) == 1.0
    assert fingerprint_halves_identical(u) == 0.0


def test_pandas_hash_key_ignored_for_numeric_columns():
    """The mechanism behind the previous test, demonstrated on toy data."""
    num = pd.DataFrame({"a": [1.0, 2.0, float("nan")], "b": [3.0, 4.0, 5.0]})
    h = lambda df, k: pd.util.hash_pandas_object(df, index=False, hash_key=k)
    assert (h(num, "0123456789abcdef") == h(num, "fedcba9876543210")).all()
    txt = num.assign(c=["x", "y", "z"])
    assert not (h(txt, "0123456789abcdef") == h(txt, "fedcba9876543210")).all()


def test_same_fingerprints_in_every_run_of_a_role(archives, archive_manifest):
    """Every run of a dataset-role scored the identical row_id -> fingerprint assignment."""
    for (ds, role), grp in archive_manifest.groupby(["dataset", "role"]):
        ref = None
        for f in grp.archive_file:
            df = read_prediction_archive(archives / f)[["row_id", "feature_fingerprint", "y_true"]].sort_values("row_id")
            df = df.reset_index(drop=True)
            if ref is None:
                ref = df
            else:
                assert ref.equals(df), f"{ds} {role}: {f} differs"

run_tests('test_fingerprint_determinism', globals())

### `tests/test_manifest_hashes.py`

In [ ]:
# ===== tests/test_manifest_hashes.py =====
import pytest


F = ("unsw_nb15", "partition", "final_source_multiplicity_balanced")
SB = ("unsw_nb15", "partition", "source_balanced")

SELF = [
    (("cicids2017", "partition", "partition_policy_manifest.json"), "partition_policy_hash"),
    (("cicids2017", "model_selection", "selected_models_manifest.json"), "selected_models_manifest_hash"),
    (("unsw_nb15", "model_selection", "selected_models_manifest.json"), "selected_models_manifest_hash"),
    (F + ("partition_integrity_report.json",), "manifest_hash"),
    (F + ("source_multiplicity_amendment.json",), "policy_hash"),
    (F + ("implementation_fix_v3_1.json",), "implementation_note_hash"),
    (F + ("implementation_fix_v3_2.json",), "implementation_note_hash"),
    (F + ("implementation_fix_v3_3.json",), "implementation_note_hash"),
    (SB + ("diagnostic_manifest_00C_block_decomposition.json",), "diagnostic_manifest_hash"),
    (SB + ("diagnostic_manifest_00D_mixed_label.json",), "diagnostic_manifest_hash"),
]


@pytest.mark.parametrize("path,field", SELF, ids=[p[-1] for p, _ in SELF])
def test_manifest_self_hash(path, field):
    m = load_json(art(*path))
    assert manifest_self_hash(m, field) == m[field]


def test_implementation_fix_v3_4_hash_is_referenced_by_integrity_report():
    """The v3.4 note hash is not a self-hash of the stored file, but it is the hash cited by the v3.4 report."""
    fix = load_json(art(*F, "implementation_fix_v3_4.json"))
    integ = load_json(art(*F, "partition_integrity_report.json"))
    assert fix["implementation_note_hash"] == integ["implementation_note_hash"]
    assert manifest_self_hash(fix, "implementation_note_hash") != fix["implementation_note_hash"]


@pytest.mark.parametrize("ds", ["cicids2017", "unsw_nb15"])
def test_config_hashes(ds):
    sel = load_json(art(ds, "model_selection", "selected_models_manifest.json"))
    for m in ("AE", "DAE"):
        assert config_hash(sel[f"selected_{m}"]["effective_config"]) == sel[f"selected_{m}"]["config_hash"]
        top = load_json(art(ds, "model_selection", f"top_three_configs_{m}.json"))
        assert len(top["configs"]) == 3
        for c in top["configs"]:
            assert config_hash(c["effective_config"]) == c["config_hash"]
        ranking = read_csv(art(ds, "model_selection", "selection_ranking.csv"))
        assert set(c["config_hash"] for c in top["configs"]) == set(ranking[ranking.model_type == m].config_hash)


def test_smoke_config_hashes():
    for p in [SB + ("smoke_manifest_source_balanced.json",), F + ("smoke_manifest_final.json",), F + ("smoke_effective_config.json",)]:
        m = load_json(art(*p))
        assert config_hash(m["effective_config"]) == m["config_hash"]


def test_shared_contract_hashes_are_identical_across_datasets_and_stages():
    """The five protocol contracts (configs/README.md) carry the same hash in every downstream manifest."""
    keys = [k for k in load_json(art("cicids2017", "model_selection", "selected_models_manifest.json"))["upstream_identity"] if k.startswith("contract::")]
    assert len(keys) == 5
    ref = load_json(art("cicids2017", "model_selection", "selected_models_manifest.json"))["upstream_identity"]
    for p in [("unsw_nb15", "model_selection", "selected_models_manifest.json"), F + ("smoke_manifest_final.json",),
              SB + ("smoke_manifest_source_balanced.json",), SB + ("diagnostic_manifest_00C_block_decomposition.json",)]:
        ui = load_json(art(*p))["upstream_identity"]
        assert all(ui[k] == ref[k] for k in keys)
    assert load_json(art(*SB, "diagnostic_manifest_00D_mixed_label.json"))["threshold_contract_hash"] == ref["contract::threshold_contract.json"]


def test_master_dataset_hash_is_consistent():
    for ds in ("cicids2017", "unsw_nb15"):
        h = load_json(art(ds, "fingerprints", "fingerprint_audit.json"))["master_dataset_sha256"]
        assert load_json(art(ds, "model_selection", "selected_models_manifest.json"))["upstream_identity"]["master_dataset_sha256"] == h
    c = load_json(art("cicids2017", "fingerprints", "fingerprint_audit.json"))["master_dataset_sha256"]
    for p in [("cicids2017", "partition", "partition_policy_manifest.json"), ("cicids2017", "provenance", "prepartition_readiness_report.json")]:
        m = load_json(art(*p))
        assert m.get("master_dataset_sha256", m.get("upstream_hashes", {}).get("master_dataset_sha256")) == c


def test_artifact_provenance_checksums():
    prov = read_csv(ARTIFACTS / "ARTIFACT_PROVENANCE.csv")
    for r in prov.itertuples():
        assert sha256_file(ARTIFACTS.parent / r.repo_path) == r.sha256, r.repo_path

run_tests('test_manifest_hashes', globals())

### `tests/test_metrics_and_statistics.py`

In [ ]:
# ===== tests/test_metrics_and_statistics.py =====
import numpy as np
import pandas as pd
import pytest



def test_mcc_and_rates_from_frozen_counts():
    for ds in ("cicids2017", "unsw_nb15"):
        fin = read_csv(art(ds, "final_evaluation", "final_seed_results.csv"))
        for r in fin.itertuples():
            m = threshold_metrics(r.TP, r.FP, r.FN, r.TN)
            assert np.isclose(m["MCC"], r.MCC, rtol=0, atol=1e-12)
            assert np.isclose(m["Recall"], r.Recall, rtol=0, atol=1e-12)
            assert np.isclose(m["FP_per_1000_benign"], r.FP_per_1000_benign, rtol=0, atol=1e-10)
            assert np.isclose(m["Balanced_Accuracy"], r.Balanced_Accuracy, rtol=0, atol=1e-12)


def test_toy_metrics():
    y = np.array([0, 0, 0, 1, 1, 1])
    s = np.array([0.1, 0.2, 0.9, 0.3, 0.8, 0.95])
    p = apply_threshold(s, 0.5)
    m = all_metrics(y, p, s)
    assert (m["TP"], m["FP"], m["FN"], m["TN"]) == (2, 1, 1, 2)
    assert np.isclose(m["MCC"], mcc_from_counts(2, 1, 1, 2)) and np.isclose(m["MCC"], 1 / 3)
    assert apply_threshold(np.array([0.5]), 0.5)[0] == 0      # strict inequality


def test_exact_enumeration_size_and_mean():
    d = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
    e = exact_bootstrap_means(d)
    assert e.size == 3125 and np.isclose(e.mean(), d.mean())


@pytest.mark.parametrize("ds", ["cicids2017", "unsw_nb15"])
def test_paired_summary_matches_frozen_nb08(ds):
    pv = read_csv(art(ds, "statistical_analysis", "paired_seed_values.csv"))
    fb = read_csv(art(ds, "statistical_analysis", "paired_seed_bootstrap_effects.csv")).set_index("metric")
    for m in fb.index:
        d = paired_differences(pv, m).to_numpy()
        s = summarise(d)
        assert np.isclose(s["mean_diff"], fb.loc[m, "mean_difference"], rtol=0, atol=1e-12)
        assert np.isclose(s["median_diff"], fb.loc[m, "median_difference"], rtol=0, atol=1e-12)
        assert np.isclose(s["sd_diff"], fb.loc[m, "paired_difference_SD"], rtol=1e-9)
        assert np.isclose(s["dz"], fb.loc[m, "cohen_dz"], rtol=1e-9)


def test_exact_and_frozen_intervals_and_the_two_discrepancies():
    """At the six decimal places printed in Table 10, every frozen NB-08 interval bound equals the
    exact-enumeration bound except two UNSW-NB15 bounds, which the manuscript corrects (Table 10, notes 1-2).
    (The frozen UNSW-NB15 precision upper bound also differs, by 3.8e-7, which does not change its printed value.)"""
    bad = []
    for ds in ("cicids2017", "unsw_nb15"):
        pv = read_csv(art(ds, "statistical_analysis", "paired_seed_values.csv"))
        fb = read_csv(art(ds, "statistical_analysis", "paired_seed_bootstrap_effects.csv")).set_index("metric")
        for m in fb.index:
            lo, hi = exact_interval(paired_differences(pv, m).to_numpy())
            for side, ex, fr in (("lo", lo, fb.loc[m, "CI_lower"]), ("hi", hi, fb.loc[m, "CI_upper"])):
                if round(ex, 6) != round(fr, 6):
                    bad.append((ds, m, side, round(ex, 6), round(fr, 6)))
    assert sorted(bad) == [("unsw_nb15", "FP_per_1000_benign", "hi", 0.732541, 0.708504),
                           ("unsw_nb15", "Precision", "lo", -0.001263, -0.001201)]


def test_monte_carlo_bootstrap_is_close_to_exact():
    pv = read_csv(art("unsw_nb15", "statistical_analysis", "paired_seed_values.csv"))
    d = paired_differences(pv, "MCC").to_numpy()
    assert np.allclose(bootstrap_interval(d), exact_interval(d), rtol=0, atol=1e-6)

run_tests('test_metrics_and_statistics', globals())

### `tests/test_no_fingerprint_overlap.py`

In [ ]:
# ===== tests/test_no_fingerprint_overlap.py =====
import pandas as pd
import pytest


F = ("unsw_nb15", "partition", "final_source_multiplicity_balanced")


@pytest.mark.parametrize("path", [("cicids2017", "partition", "partition_membership_first100k.csv"),
                                  F + ("partition_membership_first100k.csv",)])
def test_membership_sample_has_no_crossing(path):
    assert membership_checks(read_csv(art(*path)))["fingerprints_in_more_than_one_state"] == 0


def test_integrity_reports_record_zero_crossings():
    u = load_json(art(*F, "partition_integrity_report.json"))
    assert u["status"] == "PASS" and u["gates"]["fingerprint_crossings"] == 0
    c = load_json(art("cicids2017", "partition", "partition_integrity_report.json"))
    chk = {x["check"]: x for x in c["hard_validation_results"]}
    assert chk["no_feature_fingerprint_crosses_membership_states"]["passed"]
    assert chk["no_feature_fingerprint_crosses_membership_states"]["detail"] == "crossing_fingerprints=0"


def test_development_and_final_archives_are_fingerprint_disjoint(archives, archive_manifest):
    for ds, grp in archive_manifest.groupby("dataset"):
        fps = {}
        for role in ("development", "final_test"):
            f = grp[grp.role == role].archive_file.iloc[0]
            fps[role] = set(read_prediction_archive(archives / f).feature_fingerprint)
        assert not (fps["development"] & fps["final_test"]), ds


def test_mixed_groups_only_in_development_archives():
    for ds in ("cicids2017", "unsw_nb15"):
        comp = read_csv(art(ds, "fingerprints", "derived_from_archives", "mixed_label_group_composition.csv"))
        assert set(comp.role) == {"development"}
        fin = read_csv(art(ds, "final_evaluation", "derived_from_archives", "run_summary_final_test.csv"))
        assert (fin.n_mixed_label_fingerprints == 0).all()
        audit = load_json(art(ds, "fingerprints", "fingerprint_audit.json"))
        assert len(comp) == audit["number_of_mixed_label_fingerprint_groups"]
        assert comp.rows.sum() == audit["number_of_observations_in_mixed_label_groups"]

run_tests('test_no_fingerprint_overlap', globals())

### `tests/test_notebooks_in_sync.py`

In [ ]:
# ===== tests/test_notebooks_in_sync.py =====
import json

import pytest


NB_DIR = REPO_ROOT / "notebooks" / "reproduction"
NOTEBOOKS = sorted(NB_DIR.glob("R0*.ipynb"))


def _meta(nb):
    return json.load(open(nb, encoding="utf-8"))["metadata"]["nids_reproduction"]["source_sha256"]


@pytest.mark.parametrize("nb", NOTEBOOKS, ids=lambda p: p.name)
def test_notebook_matches_sources(nb):
    for rel, h in _meta(nb).items():
        assert sha256_file(REPO_ROOT / rel) == h, f"{nb.name}: {rel} changed - run tools/build_colab_notebooks.py"


def test_every_py_file_has_a_notebook_version():
    covered = set().union(*(_meta(nb) for nb in NOTEBOOKS))
    expected = {p.relative_to(REPO_ROOT).as_posix() for d in ("src", "scripts", "tests") for p in (REPO_ROOT / d).rglob("*.py")
                if p.name not in ("__init__.py", "conftest.py", "test_notebooks_in_sync.py")}
    assert len(NOTEBOOKS) == 8
    assert expected <= covered, sorted(expected - covered)

run_tests('test_notebooks_in_sync', globals())

### `tests/test_partition_accounting.py`

In [ ]:
# ===== tests/test_partition_accounting.py =====
import numpy as np


F = ("unsw_nb15", "partition", "final_source_multiplicity_balanced")


def _gate():
    return read_csv(art(*F, "multiplicity_gate.csv"))


def test_unsw_role_capacities_and_tolerances():
    r = role_capacity_checks(_gate(), load_json(art(*F, "partition_integrity_report.json")))
    assert r["role_totals_equal_capacities"] and r["total_equals_pure_benign_rows"] and r["all_cells_pass"]


def test_unsw_source_tables_reconcile_with_gate():
    r = source_checks(read_csv(art(*F, "source_gate_actual.csv")), read_csv(art(*F, "singleton_source_gate.csv")),
                      read_csv(art(*F, "source_x_multiplicity_gate.csv")), _gate())
    assert r["source_totals_equal_gate_totals"] and r["singleton_source_equals_gate_band1"]
    assert r["source_x_band_sums_equal_gate"] and r["nonexempt_cells_pass"]


def test_unsw_ge100_band_report():
    r = band_ge100_checks(read_csv(art(*F, "band_ge100_role_report.csv")), _gate())
    assert r["rows_equal_gate"] and r["groups_total"] == 276 and r["rows_total"] == 51_479


def test_unsw_pure_benign_rows_reconcile():
    r = pure_benign_reconciliation(_gate(), read_csv(art("unsw_nb15", "fingerprints", "duplicate_group_manifest.csv")))
    integ = load_json(art(*F, "partition_integrity_report.json"))
    assert r["reconciles"]
    assert r["pure_benign_rows"] == integ["pure_benign_rows"] and r["pure_benign_fingerprints"] == integ["pure_benign_fingerprints"]


def test_unsw_tv_distances_small_after_amendment():
    rows = gate_rows(_gate())
    pool = rows.sum(axis=0)
    assert max(tv_distance(rows.loc[r], pool) for r in ROLES) < 2e-4


def test_unsw_threshold_is_calibration_q99():
    lad = read_csv(art(*F, "benign_quantile_ladder.csv"))
    smoke = load_json(art(*F, "smoke_manifest_final.json"))
    q99 = float(lad.loc[lad["quantile"] == 0.99, "calibration_benign"].iloc[0])
    assert np.isclose(q99, smoke["threshold"], rtol=1e-12, atol=0)


def test_cicids_benign_and_family_tables():
    ba = read_csv(art("cicids2017", "partition", "benign_allocation_table.csv"))
    am = load_json(art("cicids2017", "partition", "attack_allocation_manifest.json"))
    assert ba.actual_benign_count.sum() == am["B_total"] == 2_273_097
    assert (ba.absolute_row_deviation == 0).all()
    assert dict(zip(ba.partition, ba.actual_benign_count)) == am["benign_achieved_counts"]
    fam = read_csv(art("cicids2017", "partition", "family_allocation_table.csv"))
    assert fam.total_row_support.sum() == am["A_total"]
    assert am["A_dev_achieved"] + am["A_test_achieved"] + am["attacks_reserved"] == am["A_total"]


def test_cicids_role_sizes_match_handoff_and_archives():
    h = load_json(art("cicids2017", "preprocessing", "preprocessing_handoff_manifest.json"))["partition_row_counts"]
    fin = read_csv(art("cicids2017", "final_evaluation", "derived_from_archives", "run_summary_final_test.csv"))
    dev = read_csv(art("cicids2017", "model_selection", "derived_from_archives", "run_summary_development.csv"))
    assert (fin.n_rows == h["final_test"]).all() and (dev.n_rows == h["development"]).all()
    assert sum(h.values()) == 2_830_743

run_tests('test_partition_accounting', globals())

### `tests/test_preprocessing_fit_scope.py`

In [ ]:
# ===== tests/test_preprocessing_fit_scope.py =====
F = ("unsw_nb15", "partition", "final_source_multiplicity_balanced")


def test_cicids_fit_scope():
    h = load_json(art("cicids2017", "preprocessing", "preprocessing_handoff_manifest.json"))
    r = fit_scope(h)
    assert r["fit_partition"] == "benign_weight_train" and r["fit_on_any_other_partition"] is False
    assert r["final_test_fit_prohibited"]
    pol = load_json(art("cicids2017", "partition", "partition_policy_manifest.json"))
    assert pol["preprocessing_handoff_rule"]["fit"] == ["benign_weight_train"]
    assert h["partition_policy_hash"] == pol["partition_policy_hash"]


def test_cicids_model_view_is_the_fingerprinted_view():
    h = load_json(art("cicids2017", "preprocessing", "preprocessing_handoff_manifest.json"))
    a = load_json(art("cicids2017", "fingerprints", "fingerprint_audit.json"))
    assert cicids_column_order_matches(h, a)
    assert len(h["model_input_column_order"]) == 78 and h["categorical_model_columns"] == []


def test_unsw_model_view_is_schema_minus_identifiers():
    schema = read_csv(art("unsw_nb15", "provenance", "feature_schema_1_to_49.csv"))
    a = load_json(art("unsw_nb15", "fingerprints", "fingerprint_audit.json"))
    assert len(schema) == 49
    assert unsw_model_view_from_schema(schema) == a["algorithm"]["column_order"]
    assert len(a["algorithm"]["column_order"]) == 41      # 38 numerical + proto, state, service


def test_final_test_not_accessed_by_gates_or_diagnostics():
    for p in [F + ("smoke_manifest_final.json",),
              ("unsw_nb15", "partition", "source_balanced", "diagnostic_manifest_00C_block_decomposition.json"),
              ("unsw_nb15", "partition", "source_balanced", "diagnostic_manifest_00D_mixed_label.json")]:
        m = load_json(art(*p))
        assert m.get("final_test_features_accessed") is False and m.get("final_test_outcomes_accessed") is False


def test_unsw_preprocessing_refitted_on_amended_partition():
    """Every preprocessing identity hash changed between the source-balanced and the v3.4 regime, and the
    v3.4 values are the ones the selected models used."""
    keys = ["preprocessing_policy_hash", "fitted_preprocessor_hash", "feature_order_hash", "transformed_representation_manifest_hash"]
    sb = load_json(art("unsw_nb15", "partition", "source_balanced", "smoke_manifest_source_balanced.json"))["upstream_identity"]
    fn = load_json(art(*F, "smoke_manifest_final.json"))["upstream_identity"]
    sel = load_json(art("unsw_nb15", "model_selection", "selected_models_manifest.json"))["upstream_identity"]
    assert all(sb[k] != fn[k] for k in keys)
    assert all(fn[k] == sel[k] for k in keys)

run_tests('test_preprocessing_fit_scope', globals())

### `tests/test_reproduction_scripts.py`

In [ ]:
# ===== tests/test_reproduction_scripts.py =====
# ===== tests/test_reproduction_scripts.py (notebook version) =====
# The .py version runs every scripts/*.py with a subprocess. In the notebook version the reproduction
# notebooks R01-R05 are run first (R00 runs them all) and this cell checks what they wrote.
import json
ROOT = REPO_ROOT
SCRIPTS = ["reproduce_fingerprints", "reproduce_partitions", "reproduce_transfer_analysis", "reproduce_final_statistics"]


@pytest.mark.parametrize("script", SCRIPTS)
def test_script_ran_and_passed(script):
    log = pd.read_csv(ROOT / "results" / "claims_check" / f"{script}.csv")
    assert (log.status != "FAIL").all()
    assert (log.status == "PASS").sum() > 20


def test_figures_written():
    for f in ["Figure_3_1_rows_vs_fingerprints", "Figure_3_2_benign_transfer_progression", "Figure_3_3_fp_rate_by_multiplicity_band",
              "Figure_3_4_quantile_transfer", "Figure_3_5_paired_metric_differences"]:
        assert (ROOT / "results" / "figures" / f"{f}.png").stat().st_size > 10_000


def test_figure_values_equal_the_manuscript_figure_log():
    """Values plotted from results/figure_source_data equal the log written when the submitted
    manuscript figures (manuscript/figures/Figure_4-8.pdf) were drawn."""
    import json
    log = json.load(open(ROOT / "manuscript" / "figures" / "figure_values_log.json"))
    src = ROOT / "results" / "figure_source_data"
    f1 = pd.read_csv(src / "Figure_3_1_data.csv")
    assert f1.rows.tolist() == log["fig4"]["rows"] and f1.fingerprint_groups.tolist() == log["fig4"]["groups"]
    f3 = pd.read_csv(src / "Figure_3_3_data.csv")
    assert f3.rows.tolist() == log["fig5"]["rows"] and f3.groups.tolist() == log["fig5"]["groups"]
    assert [round(x, 4) for x in f3.row_rate_pct] == log["fig5"]["rate_pct"]
    f2 = pd.read_csv(src / "Figure_3_2_data.csv")
    assert abs(f2.development_benign_exceedance_pct[1] - log["fig6"]["source_balanced"]) < 1e-9
    assert abs(f2.development_benign_exceedance_pct[2] - log["fig6"]["final"]) < 1e-9
    f4 = pd.read_csv(src / "Figure_3_4_data.csv")
    assert [round(x, 4) for x in f4.dev_to_cal_ratio] == log["fig7"]["unsw_ratio"]
    f5 = pd.read_csv(src / "Figure_3_5_data.csv")
    m = {"MCC": "MCC", "Recall": "recall", "PR-AUC": "PR_AUC", "FP_per_1000_benign": "FP1000"}
    for r in f5.itertuples():
        mu, lo, hi = log["fig8"][f"{r.dataset}:{m[r.metric]}"]
        assert (round(r.mean, 6), round(r.lo, 6), round(r.hi, 6)) == (mu, lo, hi)

run_tests('test_reproduction_scripts', globals())

### `tests/test_seed_register.py`

In [ ]:
# ===== tests/test_seed_register.py =====
def test_seeds_against_artefacts():
    for ds in ("cicids2017", "unsw_nb15"):
        fin = read_csv(art(ds, "final_evaluation", "final_seed_results.csv"))
        assert tuple(sorted(fin.seed.unique())) == FINAL_SEEDS
        sel = load_json(art(ds, "model_selection", "selected_models_manifest.json"))
        assert tuple(sel["confirmation_seeds"]) == CONFIRMATION_SEEDS
        b = read_csv(art(ds, "statistical_analysis", "paired_seed_bootstrap_effects.csv"))
        assert set(b.bootstrap_seed) == {BOOTSTRAP_SEED}
        assert set(b.bootstrap_iterations) == {BOOTSTRAP_ITERATIONS}
        assert set(b.confidence_level) == {CONFIDENCE_LEVEL}
        assert set(b.n_seed_pairs) == {5}
    assert load_json(art("cicids2017", "partition", "partition_policy_manifest.json"))["PARTITION_SEED"] == PARTITION_SEED
    assert load_json(art("unsw_nb15", "partition", "final_source_multiplicity_balanced",
                         "partition_integrity_report.json"))["partition_seed"] == PARTITION_SEED

run_tests('test_seed_register', globals())

## Summary

In [ ]:
summary = pd.DataFrame(TEST_RESULTS, columns=["module", "test", "case", "status", "message"])
print(summary.status.value_counts().to_string())
from IPython.display import display
display(summary[summary.status != "passed"])
assert not summary.status.isin(["FAILED", "ERROR"]).any(), "some tests failed - see the table above"